# DeepSeek-V4-Flash OpenStacks (direct pddl, sliding-window-orders unraveling data)

In [1]:
# Cell 1: Setup imports, paths, and DeepSeek runtime
from pathlib import Path
import os
import re
import json
import time
import subprocess
from datetime import datetime

from dotenv import load_dotenv
import requests

# Project root (run notebook from formalizer-scaling/src/pddl_gen_notebooks/OpenStacks/sliding_window_orders)
PROJECT_ROOT = Path.cwd().parents[3]

DATA_DIR = PROJECT_ROOT / "data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "OpenStacks" / "only_pddl_instruction.txt"
PDDL_DIR = DATA_DIR / "pddl_unraveling_domain" / "OpenStacks" / "sliding_window_orders"
DESC_DIR = DATA_DIR / "descriptions_unraveling_domain" / "OpenStacks" / "sliding_window_orders"

OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl_direct_unraveling" / "OpenStacks" / "sliding_window_orders" / "deepseek_v4_flash"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path("/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate")

# Load .env and configure DeepSeek API access
load_dotenv(PROJECT_ROOT / ".env")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise ValueError("DEEPSEEK_API_KEY not found in .env")

DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MODEL_NAME = "deepseek-v4-flash"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("PDDL dir exists:", PDDL_DIR.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/direct_pddl_direct_unraveling/OpenStacks/sliding_window_orders/deepseek_v4_flash
VAL exists: True
PDDL dir exists: True
Prompt exists: True
Model: deepseek-v4-flash


In [3]:
# Cell 2: Define target problems and load static prompt text
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 81)]  # p01..p60

PROMPT_PREFIX = PROMPT_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Description dir exists:", DESC_DIR.exists())
print("PDDL dir exists:", PDDL_DIR.exists())

Num problems: 80
First/last: p01 p80
Prompt chars: 2949
Description dir exists: True
PDDL dir exists: True


In [4]:
# Cell 3: Load NL descriptions and ground-truth PDDL for one problem ID
def load_problem_artifacts(problem_id: str):
    domain_nl_path = DESC_DIR / f"{problem_id}_domain.txt"
    problem_nl_path = DESC_DIR / f"{problem_id}_problem.txt"
    gt_domain_pddl_path = PDDL_DIR / f"{problem_id}_domain.pddl"
    gt_problem_pddl_path = PDDL_DIR / f"{problem_id}_problem.pddl"

    if not domain_nl_path.exists():
        raise FileNotFoundError(f"Missing domain NL file: {domain_nl_path}")
    if not problem_nl_path.exists():
        raise FileNotFoundError(f"Missing problem NL file: {problem_nl_path}")
    if not gt_domain_pddl_path.exists():
        raise FileNotFoundError(f"Missing ground-truth domain PDDL: {gt_domain_pddl_path}")
    if not gt_problem_pddl_path.exists():
        raise FileNotFoundError(f"Missing ground-truth problem PDDL: {gt_problem_pddl_path}")

    return {
        "nl_domain": domain_nl_path.read_text(),
        "nl_problem": problem_nl_path.read_text(),
        "gt_domain_pddl_path": gt_domain_pddl_path,
        "gt_problem_pddl_path": gt_problem_pddl_path,
        "gt_domain_pddl_text": gt_domain_pddl_path.read_text(),
    }

In [5]:
# Cell 4: Build prompt for one problem (combined primer + 3-input setup)
def build_prompt(problem_id: str):
    artifacts = load_problem_artifacts(problem_id)

    return (
        f"{PROMPT_PREFIX}\n\n"
        "You are an expert PDDL engineer.\n"
        "Create a syntactically correct and semantically consistent problem file "
        "using the information below.\n\n"
        "You are given three inputs:\n"
        "1) Original natural language domain description\n"
        "2) Original natural language problem description\n"
        "3) Ground-truth domain PDDL (fixed reference)\n\n"
        "### Original domain description\n"
        f"{artifacts['nl_domain']}\n\n"
        "### Original problem description\n"
        f"{artifacts['nl_problem']}\n\n"
        "### Ground-truth domain PDDL\n"
        f"{artifacts['gt_domain_pddl_text']}\n\n"
        "Output contract:\n"
        "- Generate ONLY the final problem PDDL.\n"
        "- Wrap it in exactly one <problem_file>...</problem_file> block.\n"
        "- Do not output any <domain_file> block.\n"
        "- Do not include any extra text before or after <problem_file>...</problem_file>."
    )

In [6]:
# Cell 5: Call DeepSeek for one prompt
def call_deepseek(prompt: str, temperature: float = 0.4, max_output_tokens: int = 220000):
    url = f"{DEEPSEEK_BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_output_tokens,
    }

    response = requests.post(url, headers=headers, json=payload, timeout=600)
    response.raise_for_status()
    data = response.json()
    return (data["choices"][0]["message"].get("content") or "").strip()

In [7]:
# Cell 6: Parse model output and extract problem_file block
def extract_problem_pddl(raw_text: str):
  if not raw_text:
      return ""

  matches = re.findall(r"<problem_file>(.*?)</problem_file>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if matches:
      return matches[-1].strip()

  return ""

In [8]:
# Cell 7: Solve one generated problem via planning.domains API (online solver)
import requests
import re

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic:
  # 1) explicit plan text exists
  # 2) solver stdout says a plan was found (covers cost-0 empty-plan case)
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          # cost-0 case: keep empty plan file so VAL can still run
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,  # can be empty for cost-0
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [9]:
# Cell 7b: Solve one generated problem via planning.domains API (EC2 self-hosted endpoint, resilient polling)
import requests
import re
from requests.exceptions import ReadTimeout, ConnectionError, ChunkedEncodingError

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 1200):
  solver_name = "dual-bfws-ffparser"
  base_url = "http://100.51.78.30:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=(10, 120),
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=(5, 180))
          poll.raise_for_status()
          res = poll.json()
      except (ReadTimeout, ConnectionError, ChunkedEncodingError, ConnectionResetError):
          print("[solver-online] transient poll error; retrying...")
          time.sleep(1.0)
          continue
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic:
  # 1) explicit plan text exists
  # 2) solver stdout says a plan was found (covers cost-0 empty-plan case)
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          # cost-0 case: keep empty plan file so VAL can still run
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("" if plan_text.endswith("\n") else "\n"))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,  # can be empty for cost-0
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [10]:
# Cell 7c: Online solver (public endpoint) + raw API JSON persistence
import requests
import re
import json

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()
  res = None

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  # Persist full final API response for debugging/truncation analysis
  (work_dir / "solver_raw_response.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic for zero-cost empty-plan cases
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [11]:
# Cell 8: Validate plan with VAL (terminal-equivalent capture, no timeout)
def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  print(f"[val] start: {' '.join(cmd)}")

  proc = subprocess.run(
      cmd,
      capture_output=True,
      text=True,
  )

  stdout = proc.stdout or ""
  stderr = proc.stderr or ""
  merged = (stdout + ("\n" + stderr if stderr else "")).strip()

  val_ok = "Plan valid" in merged
  print(f"[val] done. valid={val_ok}")
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [12]:
# Cell 9: Save generated artifacts for one problem
def save_problem_outputs(problem_id: str, raw_text: str, problem_pddl_text: str, domain_pddl_text: str, solver_result: dict, val_result: dict | None):
    problem_dir = OUTPUT_ROOT / problem_id
    cand_dir = problem_dir / "cand_00"
    cand_dir.mkdir(parents=True, exist_ok=True)

    (cand_dir / "raw.txt").write_text(raw_text or "")
    (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
    (cand_dir / "domain.pddl").write_text(domain_pddl_text or "")
    (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
    (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

    if solver_result.get("plan_found"):
        (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")
    else:
        (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")

    if val_result is not None:
        (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

    return cand_dir

In [13]:
# Cell 10: Run one problem through full pipeline
def process_one_problem(problem_id: str):
    artifacts = load_problem_artifacts(problem_id)
    prompt = build_prompt(problem_id)
    raw_text = call_deepseek(prompt)

    problem_pddl_text = extract_problem_pddl(raw_text)

    if not problem_pddl_text.strip():
        solver_result = {
            "plan_found": False,
            "plan_text": "",
            "solver_error": "No <problem_file>...</problem_file> found in model output.",
            "stdout": "",
            "stderr": "",
        }
        save_problem_outputs(problem_id, raw_text, "", artifacts["gt_domain_pddl_text"], solver_result, None)
        return {
            "problem": problem_id,
            "parse_ok": False,
            "plan_found": False,
            "val_ok": False,
            "solver_error": solver_result["solver_error"],
        }

    problem_dir = OUTPUT_ROOT / problem_id / "cand_00"
    problem_dir.mkdir(parents=True, exist_ok=True)
    gen_problem_path = problem_dir / "problem.pddl"
    gen_problem_path.write_text(problem_pddl_text)

    solver_result = run_solver(
        domain_pddl_path=artifacts["gt_domain_pddl_path"],
        problem_pddl_path=gen_problem_path,
        work_dir=problem_dir,
        timeout_sec=300,
    )

    val_result = None
    val_ok = False
    if solver_result["plan_found"]:
        plan_path = problem_dir / "plan"
        if not plan_path.exists():
            plan_path = problem_dir / "plan.txt"
            plan_path.write_text(solver_result["plan_text"])
        val_result = run_val(artifacts["gt_domain_pddl_path"], gen_problem_path, plan_path)
        val_ok = bool(val_result["val_ok"])

    save_problem_outputs(problem_id, raw_text, problem_pddl_text, artifacts["gt_domain_pddl_text"], solver_result, val_result)

    return {
        "problem": problem_id,
        "parse_ok": True,
        "plan_found": solver_result["plan_found"],
        "val_ok": val_ok,
        "solver_error": solver_result.get("solver_error", ""),
    }

In [21]:
# Cell 11: Generate only (no solver/VAL)
smoke_problem = "p10"
artifacts = load_problem_artifacts(smoke_problem)

prompt = build_prompt(smoke_problem)
raw_text = call_deepseek(prompt)
problem_pddl_text = extract_problem_pddl(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
(cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_problem_pddl_chars:", len(problem_pddl_text or ""))
print("saved:", cand_dir)

Generation-only done.
problem: p10
extracted_problem_pddl_chars: 913
saved: /home/sagemaker-user/formalizer-scaling/output/direct_pddl_direct_unraveling/OpenStacks/sliding_window_orders/deepseek_v4_flash/p10/cand_00


In [ ]:
# Cell 12: Solver+VAL only (no model call, concise output) -- VAL uses generated problem file
import contextlib
import io

smoke_problem = "p01"
artifacts = load_problem_artifacts(smoke_problem)
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

problem_path = cand_dir / "problem.pddl"
if not problem_path.exists() or not problem_path.read_text().strip():
    raise FileNotFoundError(f"Missing/empty generated problem file: {problem_path}")

with contextlib.redirect_stdout(io.StringIO()):
    solver_result = run_solver(
        domain_pddl_path=artifacts["gt_domain_pddl_path"],
        problem_pddl_path=problem_path,
        work_dir=cand_dir,
        timeout_sec=2000,
    )

(cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
(cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

produced_files = ["solver_stdout.txt", "solver_stderr.txt"]
val_result = None

if solver_result["plan_found"]:
    plan_file = cand_dir / "plan.txt"
    plan_text = solver_result.get("plan_text", "")
    plan_file.write_text(plan_text)
    produced_files.append("plan.txt")

    with contextlib.redirect_stdout(io.StringIO()):
        val_result = run_val(artifacts["gt_domain_pddl_path"], problem_path, plan_file)

    (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
    produced_files.append("validator.log")
else:
    (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")
    produced_files.append("solver_error.txt")

print("Produced files:")
for name in produced_files:
    print("-", name)
print("VAL result:", "valid" if (val_result and val_result.get("val_ok")) else ("invalid" if val_result else "not run (solver failed)"))
print("Solver summary:")
print({
    "problem": smoke_problem,
    "plan_found": solver_result["plan_found"],
    "val_ok": bool(val_result and val_result.get("val_ok")),
    "solver_error": solver_result.get("solver_error", ""),
})


In [24]:
# Cell 13: Batch generation-only (with tqdm), default p02..p60
from tqdm.auto import tqdm

BATCH_PROBLEMS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(2, 81)]  # change this list for other batches

gen_results = []
for pid in tqdm(BATCH_PROBLEMS, desc="LLM generation", unit="problem"):
    try:
        artifacts = load_problem_artifacts(pid)
        prompt = build_prompt(pid)
        raw_text = call_deepseek(prompt)
        problem_pddl_text = extract_problem_pddl(raw_text)

        cand_dir = OUTPUT_ROOT / pid / "cand_00"
        cand_dir.mkdir(parents=True, exist_ok=True)

        (cand_dir / "raw.txt").write_text(raw_text or "")
        (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
        (cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])

        ok = bool(problem_pddl_text.strip())
        gen_results.append({"problem": pid, "generation_ok": ok, "error": ""})
        print(f"{pid}: generation {'ok' if ok else 'failed (no <problem_file>)'}")
    except Exception as e:
        gen_results.append({"problem": pid, "generation_ok": False, "error": str(e)})
        print(f"{pid}: generation error -> {e}")

print("\nGeneration batch done.")
print("total:", len(gen_results))
print("ok:", sum(1 for r in gen_results if r['generation_ok']))
print("failed:", sum(1 for r in gen_results if not r['generation_ok']))


LLM generation:   0%|          | 0/79 [00:00<?, ?problem/s]

p04: generation ok


p05: generation ok


p06: generation ok


p07: generation ok


...

p76: generation ok


p77: generation ok


p78: generation ok


p79: generation ok


p80: generation ok

Generation batch done.
total: 79
ok: 79
failed: 0


In [ ]:
# Cell 14: Batch solver+VAL only (self-contained, with its own problem handle)
from tqdm.auto import tqdm
import contextlib
import io

BATCH_PROBLEMS_SOLVER_VAL = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 11)]  # change as needed

sv_results = []
for pid in tqdm(BATCH_PROBLEMS_SOLVER_VAL, desc="Solver+VAL", unit="problem"):
    artifacts = load_problem_artifacts(pid)
    cand_dir = OUTPUT_ROOT / pid / "cand_00"
    problem_path = cand_dir / "problem.pddl"

    if not problem_path.exists() or not problem_path.read_text().strip():
        msg = f"missing/empty problem file: {problem_path}"
        sv_results.append({"problem": pid, "plan_found": False, "val_ok": False, "error": msg})
        print(f"{pid}: skipped ({msg})")
        continue

    with contextlib.redirect_stdout(io.StringIO()):
        solver_result = run_solver(
            domain_pddl_path=artifacts["gt_domain_pddl_path"],
            problem_pddl_path=problem_path,
            work_dir=cand_dir,
            timeout_sec=300,
        )

    (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
    (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

    val_ok = False
    if solver_result.get("plan_found"):
        plan_file = cand_dir / "plan.txt"
        plan_file.write_text(solver_result.get("plan_text", "") or "")

        with contextlib.redirect_stdout(io.StringIO()):
            val_result = run_val(artifacts["gt_domain_pddl_path"], problem_path, plan_file)

        (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
        val_ok = bool(val_result.get("val_ok", False))
        print(f"{pid}: val completed ({'valid' if val_ok else 'invalid'})")
    else:
        (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")
        print(f"{pid}: solver failed")

    sv_results.append({
        "problem": pid,
        "plan_found": bool(solver_result.get("plan_found")),
        "val_ok": val_ok,
        "error": solver_result.get("solver_error", ""),
    })

print("\nBatch solver+VAL done.")
print("total:", len(sv_results))
print("plan found:", sum(1 for r in sv_results if r['plan_found']))
print("val ok:", sum(1 for r in sv_results if r['val_ok']))


In [ ]:
# Cell 15: In-notebook metrics summary (no persistence)
import pandas as pd

SUMMARY_PROBLEMS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 11)]

rows = []
for pid in SUMMARY_PROBLEMS:
    cand_dir = OUTPUT_ROOT / pid / "cand_00"

    solver_stdout_path = cand_dir / "solver_stdout.txt"
    solver_stderr_path = cand_dir / "solver_stderr.txt"
    solver_error_path = cand_dir / "solver_error.txt"
    plan_txt_path = cand_dir / "plan.txt"
    plan_path = cand_dir / "plan"
    val_log_path = cand_dir / "validator.log"

    solver_stdout = solver_stdout_path.read_text() if solver_stdout_path.exists() else ""
    solver_stderr = solver_stderr_path.read_text() if solver_stderr_path.exists() else ""
    solver_error = solver_error_path.read_text().strip() if solver_error_path.exists() else ""

    has_plan_txt = plan_txt_path.exists() and plan_txt_path.read_text().strip() != ""
    has_plan = plan_path.exists() and plan_path.read_text().strip() != ""
    plan_found = bool(has_plan_txt or has_plan)

    val_log = val_log_path.read_text() if val_log_path.exists() else ""
    val_ran = val_log_path.exists() and val_log.strip() != ""
    val_ok = "Plan valid" in val_log

    if val_ok:
        status = "plan_valid"
    elif plan_found and val_ran:
        status = "plan_invalid"
    else:
        status = "solver_failed"

    rows.append({
        "problem_id": pid,
        "status": status,
        "plan_found": plan_found,
        "val_ran": val_ran,
        "val_ok": val_ok,
        "solver_error": solver_error,
        "solver_stdout_chars": len(solver_stdout),
        "solver_stderr_chars": len(solver_stderr),
        "cand_dir": str(cand_dir),
    })

details_df = pd.DataFrame(rows).sort_values("problem_id").reset_index(drop=True)

total = len(details_df)
summary_df = pd.DataFrame([{
    "total": total,
    "plan_found": int(details_df["plan_found"].sum()) if total else 0,
    "val_ok": int(details_df["val_ok"].sum()) if total else 0,
    "solver_failed": int((details_df["status"] == "solver_failed").sum()) if total else 0,
}])

print("Summary")
display(summary_df)
print("\nDetails")
display(details_df)

SUMMARY_DF = summary_df
DETAILS_DF = details_df
SUMMARY_PROBLEMS_USED = SUMMARY_PROBLEMS


In [ ]:
# Cell 16: Persist summary/details to CSV (run only when you want to save)
from datetime import datetime

SAVE_LABEL = "deepseek_v4_flash_batch"

METRIC_DIR = OUTPUT_ROOT / "_metrics"
METRIC_DIR.mkdir(parents=True, exist_ok=True)

if "SUMMARY_DF" not in globals() or "DETAILS_DF" not in globals():
    raise RuntimeError("Run Cell 15 first to populate SUMMARY_DF and DETAILS_DF.")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
problems_tag = f"{SUMMARY_PROBLEMS_USED[0]}_to_{SUMMARY_PROBLEMS_USED[-1]}" if len(SUMMARY_PROBLEMS_USED) > 1 else SUMMARY_PROBLEMS_USED[0]

summary_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_summary_{ts}.csv"
details_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_details_{ts}.csv"

SUMMARY_DF.to_csv(summary_out, index=False)
DETAILS_DF.to_csv(details_out, index=False)

print("Saved:")
print("-", summary_out)
print("-", details_out)


# Parser for Problem Descriptions & Difference Checking

In [22]:
# Cell 17: Diff section setup (independent, PDDL-vs-PDDL for unraveling)
from pathlib import Path
import re
import json

PROJECT_ROOT = Path.cwd().parents[3]  # run notebook from formalizer-scaling/src/pddl_gen_notebooks/OpenStacks/sliding_window_orders
PDDL_DIR = PROJECT_ROOT / "data" / "pddl_unraveling_domain" / "OpenStacks" / "sliding_window_orders"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl_direct_unraveling" / "OpenStacks" / "sliding_window_orders" / "deepseek_v4_flash"

# Single-problem handle
DIFF_PROBLEM = "p10"  # edit as needed

# Input files
GT_PDDL_PATH = PDDL_DIR / f"{DIFF_PROBLEM}_problem.pddl"
LLM_PDDL_PATH = OUTPUT_ROOT / DIFF_PROBLEM / "cand_00" / "problem.pddl"

# Output files
CAND_DIR = OUTPUT_ROOT / DIFF_PROBLEM / "cand_00"
CAND_DIR.mkdir(parents=True, exist_ok=True)

DIFF_JSON_OUT = CAND_DIR / "gt_vs_llm_diff.json"
DIFF_TXT_OUT = CAND_DIR / "gt_vs_llm_diff.txt"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Problem:", DIFF_PROBLEM)
print("GT PDDL exists:", GT_PDDL_PATH.exists())
print("LLM PDDL exists:", LLM_PDDL_PATH.exists())
print("Output dir:", CAND_DIR)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
Problem: p10
GT PDDL exists: True
LLM PDDL exists: True
Output dir: /home/sagemaker-user/formalizer-scaling/output/direct_pddl_direct_unraveling/OpenStacks/sliding_window_orders/deepseek_v4_flash/p10/cand_00


In [16]:
# Cell 18: Helper 1 - normalize whitespace and extract top-level sections
import re

def _norm_space(s: str) -> str:
  return re.sub(r"\s+", " ", s.strip())


def _extract_section_body(pddl_text: str, section_name: str) -> str:
  marker = f"(:{section_name}"
  start = pddl_text.find(marker)
  if start == -1:
      raise ValueError(f"PDDL missing :{section_name} section.")

  i = start + len(marker)
  depth = 1
  body_chars = []

  while i < len(pddl_text) and depth > 0:
      ch = pddl_text[i]
      if ch == "(":
          depth += 1
      elif ch == ")":
          depth -= 1
          if depth == 0:
              break
      body_chars.append(ch)
      i += 1

  if depth != 0:
      raise ValueError(f"Unbalanced parentheses while reading :{section_name} section.")

  return "".join(body_chars)

In [17]:
# Cell 19: Helper 2 - extract OpenStacks objects/init/goal from a PDDL problem file
_PREDICATE_RE = re.compile(r"\((next-count|stacks-avail|waiting|includes|shipped)\s+([^()]+?)\)")


def extract_openstacks_problem_structure(pddl_text: str):
  objects_text = _extract_section_body(pddl_text, "objects")
  init_text = _extract_section_body(pddl_text, "init")
  goal_text = _extract_section_body(pddl_text, "goal")

  count_objects = []
  for chunk in re.finditer(r"([^()]+?)-\s*count", objects_text, flags=re.IGNORECASE | re.DOTALL):
      count_objects.extend(_norm_space(chunk.group(1)).split())

  init_facts = []
  for m in _PREDICATE_RE.finditer(init_text):
      pred = m.group(1)
      args = _norm_space(m.group(2)).split()
      init_facts.append(tuple([pred] + args))

  goal_facts = []
  for m in _PREDICATE_RE.finditer(goal_text):
      pred = m.group(1)
      args = _norm_space(m.group(2)).split()
      goal_facts.append(tuple([pred] + args))

  return {
      "count_objects": count_objects,
      "init_facts": init_facts,
      "goal_facts": goal_facts,
  }

In [18]:
# Cell 20: Helper 3 - OpenStacks, order-insensitive structural diff (GT vs LLM)
def diff_fact_sections(gt_pddl_text: str, llm_pddl_text: str):
  gt = extract_openstacks_problem_structure(gt_pddl_text)
  llm = extract_openstacks_problem_structure(llm_pddl_text)

  gt_objects = set(gt["count_objects"])
  llm_objects = set(llm["count_objects"])
  gt_init = set(gt["init_facts"])
  llm_init = set(llm["init_facts"])
  gt_goal = set(gt["goal_facts"])
  llm_goal = set(llm["goal_facts"])

  return {
      "objects_missing_in_llm": sorted(gt_objects - llm_objects),
      "objects_extra_in_llm": sorted(llm_objects - gt_objects),
      "init_missing_in_llm": sorted(gt_init - llm_init),
      "init_extra_in_llm": sorted(llm_init - gt_init),
      "goal_missing_in_llm": sorted(gt_goal - llm_goal),
      "goal_extra_in_llm": sorted(llm_goal - gt_goal),
      "objects_match_count": len(gt_objects & llm_objects),
      "init_match_count": len(gt_init & llm_init),
      "goal_match_count": len(gt_goal & llm_goal),
      "objects_total_gt": len(gt_objects),
      "objects_total_llm": len(llm_objects),
      "init_total_gt": len(gt_init),
      "init_total_llm": len(llm_init),
      "goal_total_gt": len(gt_goal),
      "goal_total_llm": len(llm_goal),
      "perfect_match": (
          gt_objects == llm_objects and
          gt_init == llm_init and
          gt_goal == llm_goal
      ),
  }

In [19]:
# Cell 21: Helper 4 - concise mismatch-only diff report text
def build_diff_report_text(problem_id: str, diff_result: dict):
  lines = []
  lines.append(f"Problem: {problem_id}")
  lines.append(f"Perfect match: {diff_result['perfect_match']}")
  lines.append("")

  lines.append(f"Count objects missing in LLM ({len(diff_result['objects_missing_in_llm'])}):")
  for obj in diff_result["objects_missing_in_llm"]:
      lines.append(f"- {obj}")

  lines.append("")
  lines.append(f"Count objects extra in LLM ({len(diff_result['objects_extra_in_llm'])}):")
  for obj in diff_result["objects_extra_in_llm"]:
      lines.append(f"- {obj}")

  lines.append("")
  lines.append(f"Init missing in LLM ({len(diff_result['init_missing_in_llm'])}):")
  for fact in diff_result["init_missing_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Init extra in LLM ({len(diff_result['init_extra_in_llm'])}):")
  for fact in diff_result["init_extra_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Goal missing in LLM ({len(diff_result['goal_missing_in_llm'])}):")
  for fact in diff_result["goal_missing_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Goal extra in LLM ({len(diff_result['goal_extra_in_llm'])}):")
  for fact in diff_result["goal_extra_in_llm"]:
      lines.append(f"- {fact}")

  return "\n".join(lines) + "\n"

In [23]:
# Cell 22: Diff run (single problem handle, persists outputs to pXXX/cand_00)
if not GT_PDDL_PATH.exists():
  raise FileNotFoundError(f"Missing ground-truth PDDL: {GT_PDDL_PATH}")
if not LLM_PDDL_PATH.exists():
  raise FileNotFoundError(f"Missing LLM-generated PDDL: {LLM_PDDL_PATH}")

gt_pddl_text = GT_PDDL_PATH.read_text()
llm_pddl_text = LLM_PDDL_PATH.read_text()

diff_result = diff_fact_sections(gt_pddl_text, llm_pddl_text)
diff_text = build_diff_report_text(DIFF_PROBLEM, diff_result)

DIFF_JSON_OUT.write_text(json.dumps(diff_result, indent=2))
DIFF_TXT_OUT.write_text(diff_text)

print("Saved:")
print("-", DIFF_JSON_OUT)
print("-", DIFF_TXT_OUT)
print("")
print("Mismatch summary:")
print(f"- objects missing: {len(diff_result['objects_missing_in_llm'])}")
print(f"- objects extra:   {len(diff_result['objects_extra_in_llm'])}")
print(f"- init missing:    {len(diff_result['init_missing_in_llm'])}")
print(f"- init extra:      {len(diff_result['init_extra_in_llm'])}")
print(f"- goal missing:    {len(diff_result['goal_missing_in_llm'])}")
print(f"- goal extra:      {len(diff_result['goal_extra_in_llm'])}")
print(f"- perfect_match: {diff_result['perfect_match']}")

Saved:
- /home/sagemaker-user/formalizer-scaling/output/direct_pddl_direct_unraveling/OpenStacks/sliding_window_orders/deepseek_v4_flash/p10/cand_00/gt_vs_llm_diff.json
- /home/sagemaker-user/formalizer-scaling/output/direct_pddl_direct_unraveling/OpenStacks/sliding_window_orders/deepseek_v4_flash/p10/cand_00/gt_vs_llm_diff.txt

Mismatch summary:
- objects missing: 0
- objects extra:   0
- init missing:    0
- init extra:      0
- goal missing:    0
- goal extra:      0
- perfect_match: True
